# M8 Verification: Clean Rework with Baseline

M8 separates clean rework magnitude, ratio, rolling trajectory, and baseline eligibility. It is a file-provenance proxy, not semantic defect detection.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
from IPython.display import Image, display
ROOT=Path.cwd()
while ROOT != ROOT.parent and not (ROOT/'paper_v9').is_dir(): ROOT=ROOT.parent
METRICS=ROOT/'paper_v9'/'data'/'metrics'; FIGURES=ROOT/'paper_v9'/'figures'; FIGURES.mkdir(parents=True,exist_ok=True)
magnitude=pd.read_csv(METRICS/'m8_rework_magnitude.csv',dtype={'Semestre':str})
participation=pd.read_csv(METRICS/'m8_rework_participation.csv',dtype={'Semestre':str})
trajectory=pd.read_csv(METRICS/'m8_rework_trajectory.csv',dtype={'Semestre':str})
eligibility=pd.read_csv(METRICS/'m8_baseline_eligibility.csv',dtype={'Semestre':str})
metadata=json.loads((METRICS/'m8_clean_rework.metadata.json').read_text())
legacy=pd.read_csv(ROOT/'paper_v8'/'data'/'m8_rework_severity_ratio.csv',dtype={'Semestre':str})

## 1. Verification contract

M8 belongs to RQ3 and uses the same clean-path policy as M4. T3-relative windows are anchored to evaluator votes.

In [ ]:
paper=(ROOT/'paper_v8'/'latex_code'/'main.tex').read_text(encoding='utf-8')
assert paper.index(r'\subsubsection{RQ3:') < paper.index(r'\textbf{M8 --')
assert metadata['rq']=='RQ3'
assert metadata['policy_version']=='code-churn-metrics-v2'
assert len(magnitude)==14 and len(trajectory)==406
assert trajectory.groupby(['ID_Equipe','Semestre']).size().eq(29).all()
assert magnitude['clean_rework_churn_t3'].ge(0).all()
assert magnitude.loc[~magnitude['baseline_eligible_for_rework_t3'],'clean_rework_churn_t3'].eq(0).all()
assert magnitude['clean_rework_ratio_t3'].dropna().between(0,1).all()
print('M8 RQ3, policy, baseline, ratio, and trajectory contracts: PASS')

## 2. V8 to V9 traceability

The legacy ratio is audit context only. M8 v9 does not collapse magnitude and eligibility, and it does not call path provenance a defect.

In [ ]:
traceability=pd.DataFrame([
{'v8_recommendation':'Separate magnitude from ratio.','v9_decision':'Publish M8a and M8b separately.','status':'applied','evidence':'m8_rework_magnitude.csv and participation.csv','limitation_or_approval':'Ratio descriptive only.'},
{'v8_recommendation':'Condition ratio on baseline eligibility.','v9_decision':'Publish prior path count and explicit eligibility without imputation.','status':'applied','evidence':'m8_baseline_eligibility.csv','limitation_or_approval':'No baseline is not stability.'},
{'v8_recommendation':'Add late-stage rolling dynamics.','v9_decision':'Publish 29 T3-relative seven-day windows.','status':'applied','evidence':'m8_rework_trajectory.csv','limitation_or_approval':'Windows overlap.'},
{'v8_recommendation':'Use M4 clean-path policy and avoid defect claims.','v9_decision':'Use central policy and provenance terminology.','status':'applied','evidence':'metadata and producer','limitation_or_approval':'No semantic defect validation.'}
])
assert set(traceability['status'])=={'applied'}
display(traceability)
display(legacy.head())

## 3. Artifact demo

Figures consume official M8 CSV outputs.

In [ ]:
cohort=trajectory.groupby(['Semestre','window_end_day_relative_to_t3'],as_index=False).agg(team_n=('ID_Equipe','size'),total_clean_rework_churn_7d=('clean_rework_churn_7d','sum'),median_clean_rework_churn_7d=('clean_rework_churn_7d','median'),teams_with_rework_n=('clean_rework_churn_7d',lambda v:int(v.gt(0).sum())))
cohort['teams_with_rework_share']=cohort['teams_with_rework_n']/cohort['team_n']
figure=px.line(cohort,x='window_end_day_relative_to_t3',y='total_clean_rework_churn_7d',color='Semestre',markers=True,title='M8 clean rework trajectory around T3')
figure.add_vline(x=0,line_dash='dash')
figure.write_html(METRICS/'m8_clean_rework_trajectory.html',include_plotlyjs='cdn')
for ext in ('pdf','svg','png'): figure.write_image(FIGURES/f'm8_clean_rework_trajectory.{ext}',scale=2 if ext=='png' else 1)
for ext in ('pdf','svg','png'): assert (FIGURES/f'm8_clean_rework_trajectory.{ext}').is_file() and (FIGURES/f'm8_clean_rework_trajectory.{ext}').stat().st_size>0
print('M8 HTML, PDF, SVG, and PNG figures generated: PASS')

In [ ]:
display(magnitude)
display(eligibility)
display(cohort.loc[cohort['window_end_day_relative_to_t3'].isin([-21,-7,0,7])])
display(Image(filename=str(FIGURES/'m8_clean_rework_trajectory.png'),width=900))
print('M8 artifact demo: official tables and figure displayed')

## Preliminary RQ3 analysis

M8 contributes an exploratory view of changes in previously observed clean paths. It does not establish defects, destructive intent, effort, quality, or causality. Baseline-ineligible teams remain a separate state and M8 is not interpreted as stability for them.

In [ ]:
summary=magnitude.groupby('Semestre',as_index=False).agg(team_semester_n=('ID_Equipe','size'),eligible_n=('baseline_eligible_for_rework_t3','sum'),total_clean_rework_t3=('clean_rework_churn_t3','sum'),median_clean_rework_t3=('clean_rework_churn_t3','median'))
summary['analysis_level']='team_semester_then_rolling_window'
summary['inference']='exploratory_descriptive'
summary['construct_limit']='file_provenance_proxy_not_semantic_destructive_rework'
assert summary['inference'].eq('exploratory_descriptive').all()
display(summary)
print('Preliminary RQ3 reading: M8 describes baseline-conditioned clean path changes only; no defect or causal claim is supported.')